# PrecisionMiner Tutorial

This tutorial demonstrates how to use the PrecisionMiner components of the EpiScope package.  PrecisionMiner performs deep parsing and structured extraction on individual papers.  In this example we use the `LLMExtractor` to extract data source information from a set of relevant text chunks.  We also illustrate how to classify a chunk according to paper type using a simple similarity approach.  To keep the example self‑contained, heavy dependencies such as FAISS and large language models are patched out.

## Extract Data Sources with LLMExtractor

The `LLMExtractor` class wraps an LLM chat function and parses the result into structured data.  We supply a dummy chat function that returns a fixed JSON string.  The extractor then validates the structure using pydantic models and produces an `ExtractionResult`.


**Loading pdfs**

In [1]:
from episcope.ingest.document_loader import UnstructuredDocumentLoader

# add progress bar
loader = UnstructuredDocumentLoader()
results_sections = loader.load_directory("pdfs")

In [2]:
sections, metadata = results_sections['pdfs/2022_Du_k.pdf']

In [3]:
metadata.to_dict()

{'title': '2022_Du_k',
 'abstract': '',
 'authors': [],
 'publication_year': None,
 'journal': '',
 'doi': '',
 'keywords': [],
 'first_author': ''}

In [4]:
import numpy as np
ix = np.random.randint(0, len(sections))
print(ix)
sections[ix].to_dict()

26


{'title': 'a WILEY Jransboundary and Emerging Diseases',
 'content': 'Chun, B.C. (2016). Understanding and modeling the super-spreading events of the Middle East respiratory syndrome outbreak in Korea. Infection & Chemotherapy, 48, 147-149.\nDu, Z., Hong, H., Wang, S., Ma, L., Liu, C., Bai, Y., Adam, D.C., Tian, L., Wang, L., Lau, E. H. Y., & Cowling, B. J. (2022). reproduction number of the omicron variant triples that of the delta variant. Viruses, 14, 821.\nDu, Z., Javan, E., Nugent, C., Cowling, B. J., & Meyers, L. A. (2020). Using the COVID-19 to influenza ratio to estimate early pandemic spread in Wuhan, China and Seattle, US. EClinicalMedicine, 26, 100479.\nDu, Z., Liu, C., Wang, C., Xu, L., Xu, M., Wang, L., Bai, Y., Xu, X., Lau, E. H. Y., Wu, P,, & Cowling, B. J. (2022). Reproduction numbers of severe acute res- piratory syndrome coronavirus 2 (SARS-CoV-2) variants: A systematic review and meta-analysis. Clinical Infectious Diseases, 73(3), e754-e764.\nDu, Z., Tian, L., & Jin,

In [5]:
sections[10]

**Indexing**

In [6]:
list(results_sections)

['pdfs/2022_Du_k.pdf',
 'pdfs/2020_He_infectious_period.pdf',
 'pdfs/2021_Ahammed_r0.pdf']

In [11]:
from episcope.index.paper_indexer import PaperIndexer
indexer = PaperIndexer()
for paper_id in results_sections:
    sections, metadata = results_sections[paper_id]
    indexer.index_paper(sections, metadata, paper_id=paper_id)

**Retrieval**

In [15]:
indexer.search("Who are the authors of these papers?", top_k=5)

[{'text': 'Nature Research wishes to improve the reproducibility of the work that we publish. This form provides structure for consistency and transparency reporting. For further information on Nature Research policies, see Authors & Referees and the Editorial Policy Checklist.',
  'section_title': 'Reporting Summary',
  'section_type': 'Other',
  'paper_id': 'pdfs/2020_He_infectious_period.pdf',
  'is_metadata': False,
  'score': 0.6522127389907837,
  'content': 'Nature Research wishes to improve the reproducibility of the work that we publish. This form provides structure for consistency and transparency reporting. For further information on Nature Research policies, see Authors & Referees and the Editorial Policy Checklist.'},
 {'text': 'Tanvir Ahammed affirms that this manuscript is an honest, accurate, and transparent account of the study being reported; that no impor- tant aspects of the study have been omitted; and that any discrepan- cies from the study as planned (and, if rele

**Generation**

### Precision Miner

Pipeline

In [1]:
# from grobid_client.grobid_client import GrobidClient as PyClient
# grobid_url="http://192.168.1.250:8070"
# client = PyClient(grobid_url)


In [2]:
# from episcope.ingest.local_grobid_client import GrobidClient
# gc = GrobidClient(grobid_url)
# r = gc.process_fulltext("pdfs/2022_Du_k.pdf")
# meta, sec, ref = r

In [1]:
from episcope.storage.academic_db_manager import AcademicDBManager
db = AcademicDBManager()

In [2]:
from episcope.ingest.grobid_pipeline import extract_paper, extract_directory
strategy_name = "test_grobid"
storage_dir = "h_storage"
grobid_url="http://192.168.1.250:8070"

# extract_directory(
#     "pdfs/",
#     strategy_name=strategy_name,
#     db=db,
#     grobid_url=grobid_url,
#     output_dir=storage_dir
# )



# extract_paper(
#     "pdfs/2021_Ahammed_r0.pdf", 
#     strategy_name=strategy_name, 
#     db = db,
#     output_dir=storage_dir) # , grobid_url="http://localhost:8070"

In [3]:
# db._store

In [4]:
strategy_name

'test_grobid'

In [5]:
# # Could we make it load from a filesystem path?
# db.retrieve(
#     paper_id="2021_Ahammed_r0",
#     data_type="metadata",
#     strategy_name="test_grobid"
#     )

In [ ]:
from episcope.parse.processing import PipelineProcessor
# create storage directory attaching output dir from extract paper to the strategy name

storage_dirp = f"{storage_dir}/{strategy_name}"
pipeline = PipelineProcessor()
# this should not be based on the file path but rather on the paper id used in the extract paper function
result_dict, result_markdown = pipeline.process_pdf(
    "pdfs/2021_Ahammed_r0.pdf", 
    storage_path=storage_dirp, 
    strategy_name=strategy_name)

NLTK not found, using a fallback stopword list.


INFO:datasets:PyTorch version 2.6.0 available.
INFO:datasets:Polars version 1.34.0 available.
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: allenai-specter
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']
INFO:episcope.parse.indexing.embeddings:Initialized EmbeddingIndexer (compat) with paragraph chunking
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
INFO:faiss:Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes.
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: allenai-specter
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: allenai-specter
INFO:episcope.parse.processing.pipeline:Processing 2021_Ahammed_r0
INFO:root:Classification response: {
  "classification": 

tesseract 5.5.0
 leptonica-1.83.1
  libgif 5.2.1 : libjpeg 8d (libjpeg-turbo 3.0.0) : libpng 1.6.46 : libtiff 4.7.0 : zlib 1.2.13 : libwebp 1.5.0 : libopenjp2 2.5.3
 Found NEON


Processing PDF pages:  36%|███▋      | 4/11 [00:18<00:34,  4.99s/it]INFO:episcope.parse.processing.figure_table_extractor:Extracted table on page 4, region 0, table 0 with shape (6, 9)
INFO:episcope.parse.processing.figure_table_extractor:Extracted table on page 4, region 0, table 1 with shape (5, 5)
INFO:episcope.parse.processing.figure_table_extractor:Extracted table on page 4, region 0, table 2 with shape (7, 5)
Processing PDF pages:  45%|████▌     | 5/11 [00:24<00:30,  5.16s/it]INFO:episcope.parse.processing.figure_table_extractor:Extracted table on page 5, region 0, table 0 with shape (9, 4)
INFO:episcope.parse.processing.figure_table_extractor:Extracted table on page 5, region 0, table 1 with shape (3, 3)
INFO:episcope.parse.processing.figure_table_extractor:Extracted table on page 5, region 0, table 2 with shape (88, 4)


In [ ]:
# render the markdown in the cell
from IPython.display import Markdown, display
display(Markdown(result_markdown))

# Paper ID: 2021_Ahammed_r0

**Title:** 2021_Ahammed_r0

**First Author:** 

**Publication Year:** None

**DOI:** 

**Journal:** 

**Analysis Type:** literature_review

**PDF Path:** pdfs/2021_Ahammed_r0.pdf


---

## Abstract

N/A

## Keywords

N/A

## Extracted Data Sources

No data sources extracted.


---

## Data Sources Description

N/A


The way sections are stored by extract paper @episcope.ingest.grobid_pipeline (at least for unstructured version) does not aggree with how they are then loaded by PipelineProcessor @episcope.parse.processing, Fix this discrepancy (probably acting on the way they are stored, since the blueprinåts are alredy defined and used elsewhere in the code). Check that this change doesn t breack other dependencies elsewhere in the codebase

In [2]:
from episcope.parse.extraction.llm_extractor import LLMExtractor
import json

# Define a dummy chat function that returns a JSON object matching the schema.
# We use json.dumps to build the JSON string to avoid escaping issues.
def dummy_chat_fn(model_name: str, messages: list, options: dict) -> dict:
    payload = {
        'data_sources_description': 'Clinical trial database',
        'data_sources': [{
            'source_name': 'ClinicalTrials.gov',
            'url': 'https://clinicaltrials.gov',
            'explanation': 'Repository of clinical trial records',
            'section_found': 'Methods'
        }],
        'references': []
    }
    return {'content': json.dumps(payload)}

# Create an extractor with the dummy chat function
extractor = LLMExtractor(chat_fn=dummy_chat_fn)

# Minimal metadata object with title and abstract
class Meta:
    title = 'Sample Paper'
    abstract = 'This study investigates the efficacy of vaccination.'

# Define relevant text chunks (could be from a PDF)
chunks = [
    {'text': 'The study recruited 100 participants from multiple clinics.'},
    {'text': 'Data were collected from the ClinicalTrials.gov registry.'}
]

# Extract data sources
extraction_result, _ = extractor.extract_data_sources(
    relevant_chunks=chunks,
    references=[],
    paper_type='data_analysis',
    metadata=Meta(),
    query='What data sources were used in this study?'
)

# Display the result
from pprint import pprint
pprint(extraction_result)

ExtractionResult(data_sources_description='Clinical trial database', data_sources=[DataSourceItem(source_name='ClinicalTrials.gov', url='https://clinicaltrials.gov', explanation='Repository of clinical trial records', section_found='Methods')], references=[], matched_dois=[])


## Classify a Chunk by Paper Type

The `PaperClassifier` uses semantic similarity to categorize chunks into paper types (e.g., literature review, data analysis).  For this example we patch the sentence transformer with a dummy implementation that returns a vector whose value is the number of words.  We then call the internal `_classify_chunk_by_templates` method to determine the paper type.


In [2]:
# Patch the SentenceTransformer within PaperClassifier to avoid loading models
from unittest.mock import patch
import numpy as np
from episcope.parse.extraction.paper_classifier import PaperClassifier

class DummySentenceTransformer:
    def __init__(self, *args, **kwargs):
        pass
    def encode(self, texts, **kwargs):
        if isinstance(texts, list):
            return np.array([[len(t.split())] for t in texts])
        return np.array([len(texts.split())])

with patch('episcope.parse.extraction.paper_classifier.SentenceTransformer', DummySentenceTransformer):
    classifier = PaperClassifier(model_name='dummy', embedding_model='dummy', use_gpu=False)
    text = 'We analysed data from the national survey including thousands of participants.'
    paper_type = classifier._classify_chunk_by_templates(text)
    print(f'Chunk: {text} Classified as: {paper_type}')

Chunk: We analysed data from the national survey including thousands of participants. Classified as: literature_review


## Loading and Indexing a Paper

In a real PrecisionMiner workflow you will parse a PDF into structured sections, index those sections and then perform classification and extraction.  The document loader factory simplifies the first part of this process.  Below we demonstrate how to load a text file and index it with the per‑paper indexer.

In [20]:
from episcope.ingest import DocumentLoaderFactory
from episcope.index.paper_indexer import PaperIndexer
from episcope.retrieve.precision import PrecisionMinerRetriever

# Create a loader and indexer
loader = DocumentLoaderFactory.get_loader('unstructured')
indexer = PaperIndexer(min_chunk_size=5)

import tempfile
temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.txt')
temp_file.write(b'In this review we summarise several studies.\n\nData were collected from multiple sources.')
temp_file.close()

sections, metadata = loader.load(temp_file.name)
indexer.index_paper(sections, metadata, paper_id='DEMO')
# Use the PrecisionMinerRetriever with no external classifier/extractor
retriever = PrecisionMinerRetriever(indexer=indexer)
results = retriever.retrieve('What data sources are used?', paper_ids=['DEMO'])
from pprint import pprint
pprint(results[0])

ImportError: cannot import name 'DocumentLoaderFactory' from 'episcope.ingest' (/Users/vins/Documents/Projects/EpiScope/episcope/ingest/__init__.py)

## Summary

This notebook illustrated how to use key components of the PrecisionMiner pipeline without relying on external services.  We extracted structured data sources from simple text chunks and performed a basic paper type classification using a dummy embedding model.  We also showed how to load and index a document with the per‑paper indexer and run the PrecisionMinerRetriever over it.  In a full pipeline you would run the `parse/processing/pipeline.py` script to orchestrate GROBID parsing, OCR, table extraction, and the LLM extractor, saving the results in your project directory.

In [ ]:

INFO:faiss:Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes.
